# Tutor Pipeline Evaluation Report

This notebook brings together the added semantic threshold study, the tutor-policy ablation
benchmark, and the failure-case analysis used in the written report. The tables are designed
to be easy to screenshot directly from a notebook view.


In [1]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import HTML, display

ROOT = Path.cwd().resolve()
if (ROOT / "reports").exists():
    PROJECT_ROOT = ROOT
else:
    PROJECT_ROOT = ROOT.parent.parent

ANALYSIS_DIR = PROJECT_ROOT / "reports" / "analysis"
ANALYSIS_DIR


PosixPath('/home/joels/PycharmProjects/nlp-adaptive-tutor/reports/analysis')

In [2]:
semantic = pd.read_csv(ANALYSIS_DIR / "semantic_threshold_scores.csv")
semantic_summary = pd.read_csv(ANALYSIS_DIR / "semantic_group_summary.csv")
ablation = pd.read_csv(ANALYSIS_DIR / "pipeline_ablation_results.csv")
benchmark = pd.read_csv(ANALYSIS_DIR / "pipeline_benchmark_cases.csv")
failures = pd.read_csv(ANALYSIS_DIR / "failure_cases.csv")

semantic


,case_id,lesson_id,item_id,learner_text,label,expected_accept,note,target_text,tfidf_score,transformer_score,tfidf_pass_065,tfidf_pass_080,transformer_pass_065,transformer_pass_080
0,S1,1,1,Estudio NLP en la universidad hoy.,match,1,Exact target answer.,Estudio NLP en la universidad hoy.,1.000000,1.000000,1,1,1,1
1,S2,1,1,Hoy estudio NLP en la universidad.,match,1,Word-order variation with the same meaning.,Estudio NLP en la universidad hoy.,0.835050,0.901884,1,1,1,1
2,S3,1,1,Estudio procesamiento del lenguaje natural en ...,match,1,Acronym expanded into a lexical paraphrase.,Estudio NLP en la universidad hoy.,0.422233,0.520902,0,0,0,0
3,S4,3,3,Donde esta la estacion de tren?,match,1,Accent-free version of the target question.,¿Dónde está la estación de tren?,0.224375,0.979322,0,0,1,1
4,S5,4,1,Quiero un vaso con agua por favor.,match,1,Near paraphrase with one preposition change.,"Quiero un vaso de agua, por favor.",0.627904,0.976553,0,0,1,1
5,S6,4,2,Tengo mucha hambre despues de clase hoy.,match,1,Meaning preserved with an added intensifier.,Tengo hambre después de clase hoy.,0.417645,0.931294,0,0,1,1
6,S7,3,1,Voy a la library despues de clase.,match,1,"Code-switched noun, but the intent is still cl...",Voy a la biblioteca después de clase.,0.377913,0.917663,0,0,1,1
7,S8,4,1,Quiero agua por favor.,partial,0,Reduced answer that drops the 'glass of water'...,"Quiero un vaso de agua, por favor.",0.477139,0.796069,0,0,1,0
8,S9,2,2,A las siete cada dia.,partial,0,Time phrase only; the main verb is omitted.,Me levanto a las siete cada día.,0.407115,0.858985,0,0,1,1
9,S10,1,1,Tengo hambre despues de clase hoy.,mismatch,0,"Well-formed Spanish, but unrelated meaning.",Estudio NLP en la universidad hoy.,0.048185,0.429584,0,0,0,0


In [3]:
def bar_cell(value, max_value, color):
    value = float(value)
    pct = 0.0 if max_value == 0 else (100.0 * value / max_value)
    return (
        "<div style='display:flex;align-items:center;gap:8px'>"
        "<div style='width:140px;height:12px;background:#edf2f7;border-radius:999px;overflow:hidden'>"
        f"<div style='width:{pct:.1f}%;height:12px;background:{color}'></div>"
        "</div>"
        f"<span style='font-family:monospace'>{value:.3f}</span>"
        "</div>"
    )


def render_bar_table(df, value_cols, extra_cols=None, colors=None):
    extra_cols = extra_cols or []
    colors = colors or {}
    max_values = {col: max(float(df[col].max()), 1e-9) for col in value_cols}
    headers = extra_cols + value_cols
    html = ["<table style='border-collapse:collapse;width:100%;font-size:14px'>"]
    html.append("<thead><tr>")
    for col in headers:
        html.append(
            f"<th style='text-align:left;border-bottom:2px solid #cbd5e1;padding:8px'>{col}</th>"
        )
    html.append("</tr></thead><tbody>")
    for _, row in df.iterrows():
        html.append("<tr>")
        for col in extra_cols:
            html.append(
                f"<td style='padding:8px;border-bottom:1px solid #e2e8f0;vertical-align:top'>{row[col]}</td>"
            )
        for col in value_cols:
            html.append(
                "<td style='padding:8px;border-bottom:1px solid #e2e8f0'>"
                + bar_cell(row[col], max_values[col], colors.get(col, "#2563eb"))
                + "</td>"
            )
        html.append("</tr>")
    html.append("</tbody></table>")
    display(HTML("".join(html)))


def render_confusion_table(df):
    max_value = max(int(df.to_numpy().max()), 1)
    html = ["<table style='border-collapse:collapse;font-size:14px'>"]
    html.append("<thead><tr><th style='padding:8px'></th>")
    for col in df.columns:
        html.append(
            f"<th style='padding:8px;border-bottom:2px solid #cbd5e1'>{col}</th>"
        )
    html.append("</tr></thead><tbody>")
    for idx, row in df.iterrows():
        html.append("<tr>")
        html.append(
            f"<th style='padding:8px;text-align:left;border-right:2px solid #cbd5e1'>{idx}</th>"
        )
        for value in row:
            alpha = 0.12 + (float(value) / max_value) * 0.78
            html.append(
                "<td style='padding:10px;text-align:center;color:#0f172a;"
                f"background:rgba(37,99,235,{alpha:.2f});border:1px solid #ffffff'>{int(value)}</td>"
            )
        html.append("</tr>")
    html.append("</tbody></table>")
    display(HTML("".join(html)))


## Semantic Threshold Cases

The table below compares TF-IDF against the transformer score on exact matches, paraphrases,
partials, and mismatches. Cases where the two columns disagree are usually the most useful
ones to discuss in the report.


In [4]:
semantic_view = semantic[
    [
        "case_id",
        "label",
        "learner_text",
        "tfidf_score",
        "transformer_score",
    ]
].copy()

render_bar_table(
    semantic_view,
    value_cols=["tfidf_score", "transformer_score"],
    extra_cols=["case_id", "label", "learner_text"],
    colors={
        "tfidf_score": "#2563eb",
        "transformer_score": "#9333ea",
    },
)


case_id,label,learner_text,tfidf_score,transformer_score
S1,match,Estudio NLP en la universidad hoy.,1.000,1.000
S2,match,Hoy estudio NLP en la universidad.,0.835,0.902
S3,match,Estudio procesamiento del lenguaje natural en la universidad hoy.,0.422,0.521
S4,match,Donde esta la estacion de tren?,0.224,0.979
S5,match,Quiero un vaso con agua por favor.,0.628,0.977
S6,match,Tengo mucha hambre despues de clase hoy.,0.418,0.931
S7,match,Voy a la library despues de clase.,0.378,0.918
S8,partial,Quiero agua por favor.,0.477,0.796
S9,partial,A las siete cada dia.,0.407,0.859
S10,mismatch,Tengo hambre despues de clase hoy.,0.048,0.430


## Mean Similarity by Case Type

This summary is useful when you want one screenshot that explains the main semantic trade-off:
TF-IDF is conservative, while the transformer is more permissive.


In [5]:
render_bar_table(
    semantic_summary,
    value_cols=["tfidf_score", "transformer_score"],
    extra_cols=["label"],
    colors={
        "tfidf_score": "#2563eb",
        "transformer_score": "#9333ea",
    },
)


label,tfidf_score,transformer_score
match,0.558,0.890
mismatch,0.046,0.231
partial,0.442,0.828


## Tutor Ablation Results

`full` uses the real pipeline. `no_syntax` removes syntax issues before the policy decision.
`no_fluency` removes the fluency nudge signal.


In [6]:
render_bar_table(
    ablation,
    value_cols=["action_match_rate"],
    extra_cols=["variant", "correct_cases", "total_cases"],
    colors={"action_match_rate": "#059669"},
)


variant,correct_cases,total_cases,action_match_rate
full,8,8,1.000
no_syntax,6,8,0.750
no_fluency,7,8,0.875


In [7]:
benchmark[
    [
        "case_id",
        "expected_action",
        "full_action",
        "no_syntax_action",
        "no_fluency_action",
        "note",
    ]
]


,case_id,expected_action,full_action,no_syntax_action,no_fluency_action,note
0,P1,GOOD,GOOD,GOOD,GOOD,Exact answer.
1,P2,GOOD,GOOD,GOOD,GOOD,Same meaning with natural reordering.
2,P3,LANGUAGE_MISMATCH,LANGUAGE_MISMATCH,LANGUAGE_MISMATCH,LANGUAGE_MISMATCH,"Correct meaning, wrong language."
3,P4,MEANING_MISMATCH,MEANING_MISMATCH,MEANING_MISMATCH,MEANING_MISMATCH,Grammatical Spanish with unrelated content.
4,P5,MEANING_MISMATCH,MEANING_MISMATCH,MEANING_MISMATCH,MEANING_MISMATCH,Wrong lesson answer.
5,P6,SYNTAX_FIX,SYNTAX_FIX,MEANING_MISMATCH,SYNTAX_FIX,"Relevant words present, but the sentence is a ..."
6,P7,SYNTAX_FIX,SYNTAX_FIX,MEANING_MISMATCH,SYNTAX_FIX,Noun phrase instead of a full sentence.
7,P8,FLUENCY_NUDGE,FLUENCY_NUDGE,FLUENCY_NUDGE,GOOD,"Meaning is good, but the LM assigns low fluency."


## Failure Cases

These are the most report-worthy examples because they show where the current tutor is brittle:
parser sensitivity, strict TF-IDF thresholds, and the short-input gate.


In [8]:
failure_view = failures[
    [
        "case_id",
        "learner_text",
        "expected_action",
        "predicted_action",
        "issue",
        "tfidf_score",
        "transformer_score",
        "syntax_codes",
    ]
].copy()

render_bar_table(
    failure_view,
    value_cols=["tfidf_score", "transformer_score"],
    extra_cols=[
        "case_id",
        "learner_text",
        "expected_action",
        "predicted_action",
        "issue",
        "syntax_codes",
    ],
    colors={
        "tfidf_score": "#2563eb",
        "transformer_score": "#9333ea",
    },
)


case_id,learner_text,expected_action,predicted_action,issue,syntax_codes,tfidf_score,transformer_score
F1,Donde esta la estacion de tren?,GOOD,SYNTAX_FIX,Parser sensitivity to accent-free input.,"MISSING_VERB,ES_VERB_EXPECTED",0.224,0.979
F2,Me levanto a las siete cada dia.,GOOD,MEANING_MISMATCH,TF-IDF threshold rejects a near-exact answer.,nan,0.695,0.998
F3,A las siete.,GOOD,INPUT_TOO_SHORT,Short but plausible answer is blocked before scoring.,nan,nan,nan
F4,Trabajo en mi tesis este año.,GOOD,SYNTAX_FIX,Exact target is mis-flagged by syntax rules.,"MISSING_VERB,ES_VERB_EXPECTED",1.000,1.000
F5,Estudio procesamiento del lenguaje natural en la universidad hoy.,GOOD,SYNTAX_FIX,Lexical paraphrase and acronym expansion are not handled well.,"MISSING_VERB,NOUN_HEAVY_FRAGMENT,ES_VERB_EXPECTED",0.422,0.521
